Imports

In [ ]:
import os

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv

from langchain.tools import tool
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
load_dotenv()

Initilizing LLM

In [ ]:
def load_llm():
    return ChatGroq(
        model="llama-3.3-70b-versatile",  
        temperature=0,
        api_key=os.getenv("GROQ_API_KEY") 
    )

 The RAG Pipeline

In [ ]:
def load_vectorstore():
    loader = PyPDFLoader(r'C:\Users\ok\OneDrive\Documents\Agentic AI\RAG in LangGraph\intro-to-ml.pdf')
    docs = loader.load()
    len(docs)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= 500,
        chunk_overlap= 100
    )

    chunks = text_splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        encode_kwargs={'normalize_embeddings': True}
    )

    if os.path.exists('./rag_in_langgraph'):
        vectorstore = Chroma(
            persist_directory= './rag_in_langgraph',
            embedding_function= embeddings
        )
    else:
        vectorstore = Chroma.from_documents(
            documents= chunks,
            embedding= embeddings,
            persist_directory= './rag_in_langgraph'
        )
        return vectorstore
    

The RAG Tool

In [ ]:
@tool

def rag_in_langgraph(query):
    
    '''Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents'''
    retriever = load_vectorstore().as_retriever(
        search_type= 'similarity',
        search_kwargs= {'k' : 4}
    )
    
    result = retriever.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query': query,
        'context': context,
        'metadata': metadata
    }

Binding Tools to the LLM

In [ ]:
tools_list = [rag_in_langgraph]
llm_with_tools = load_llm().bind_tools(tools_list)

In [ ]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

Graph Nodes & Tool executor

In [ ]:
def chat_node(state : ChatState):
    chat = state['messages']
    messages = llm_with_tools.invoke(chat)
    return {'messages' : messages}

In [ ]:
tool_node = ToolNode(tools_list)

Graph Structure

In [ ]:
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()
chatbot

Execution 

In [ ]:
user_query = {
    'messages' : [
        HumanMessage(
            content= 'Tell me about logistic regression under 5 lines.'
        )
    ]
}

In [ ]:
response = chatbot.invoke(user_query)
for message in response['messages']:
    message.pretty_print()